# DuckDB GeoParquet Benchmarks

This notebook measures direct DuckDB queries against local STAC GeoParquet outputs.

The benchmark goal is to compare Parquet layouts, not hash generation speed. Run `scripts/sync-benchmark-data.sh` and `scripts/generate-file-count-matched-geoparquet.sh` before timing queries.

## Setup

Expected Python packages:

- `duckdb`
- `stac-hash`, only if you want to compute hash range parameters in Python

The notebook uses DuckDB's spatial extension for exact geometry queries.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import json
import statistics
import time

import duckdb
import pandas as pd

con = duckdb.connect(database=':memory:')
con.execute('INSTALL spatial')
con.execute('LOAD spatial')
con.execute('PRAGMA threads = 8')
con.execute('PRAGMA enable_object_cache = false')

duckdb.__version__

## Dataset Variants

Point each variant at a local GeoParquet file or glob. Keep all variants semantically equivalent so row counts match across layouts.

In [ ]:
DATASETS = {
    'microsoft': '../data/benchmarks/source/mspc-sentinel-2-l2a/*.parquet',
    'hashed_128_files': '../data/benchmarks/source/mspc-sentinel-2-l2a-sorted/**/*.parquet',
    'hashed_12_files': '../data/benchmarks/generated/mspc-sentinel-2-l2a-sorted-12-files/*.parquet',
}

DATASETS

## Query Parameters

Use fixed AOIs, time windows, collections, and ids so runs are comparable. Choose IDs that exist in every dataset variant. The source data currently covers 2025 Sentinel-2 L2A items.

The following cell can auto-fill `PARAMS['id']` with a real item id if the placeholder is left unchanged.


In [ ]:
PARAMS = {
    'collection': 'sentinel-2-l2a',
    'id': 'REPLACE_WITH_REAL_ITEM_ID',
    'start_datetime': datetime(2025, 6, 1, tzinfo=timezone.utc),
    'end_datetime': datetime(2025, 7, 1, tzinfo=timezone.utc),
    'minx': -109.0,
    'miny': 37.0,
    'maxx': -102.0,
    'maxy': 41.0,
    'aoi_wkt': 'POLYGON((-109 37, -102 37, -102 41, -109 41, -109 37))',
    'max_cloud_cover': 20.0,
    # Fill these once the hash range strategy is selected.
    'min_hash': 0,
    'max_hash': 9223372036854775807,
}

PARAMS

In [ ]:
# Auto-fill a real item id for the needle-in-a-haystack query.
# Override PARAMS['id'] manually above if you want a specific item.
if PARAMS['id'] == 'REPLACE_WITH_REAL_ITEM_ID':
    PARAMS['id'] = con.execute(
        "SELECT id FROM read_parquet(?, hive_partitioning = false) LIMIT 1",
        [DATASETS['microsoft']],
    ).fetchone()[0]

PARAMS['id']


## Helpers

In [ ]:
@dataclass(frozen=True)
class BenchmarkResult:
    dataset: str
    query: str
    rows: int | None
    best_seconds: float
    median_seconds: float
    runs: tuple[float, ...]


def sql_literal(value):
    if isinstance(value, datetime):
        return "TIMESTAMPTZ '" + value.isoformat().replace('+00:00', 'Z') + "'"
    if isinstance(value, str):
        return "'" + value.replace("'", "''") + "'"
    if value is None:
        return 'NULL'
    return str(value)


def render(template: str, dataset_glob: str, params: dict) -> str:
    values = {'parquet_glob': sql_literal(dataset_glob)}
    values.update({key: sql_literal(value) for key, value in params.items()})
    return template.format(**values)


def run_sql(sql: str):
    return con.execute(sql).fetchall()


def time_query(sql: str, repeats: int = 5) -> tuple[int | None, tuple[float, ...]]:
    rows = None
    timings = []
    for _ in range(repeats):
        started = time.perf_counter()
        result = con.execute(sql).fetchall()
        timings.append(time.perf_counter() - started)
        if len(result) == 1 and len(result[0]) == 1 and isinstance(result[0][0], int):
            rows = result[0][0]
        else:
            rows = len(result)
    return rows, tuple(timings)


def explain_analyze(sql: str) -> str:
    rows = con.execute('EXPLAIN ANALYZE ' + sql).fetchall()
    return '\n'.join(str(row[1] if len(row) > 1 else row[0]) for row in rows)


def explain_analyze_json(sql: str):
    rows = con.execute('EXPLAIN (ANALYZE, FORMAT json) ' + sql).fetchall()
    payload = rows[0][1] if len(rows[0]) > 1 else rows[0][0]
    return json.loads(payload)

## Parquet Metadata

Run this before timing queries. It verifies file counts, row groups, and whether the important columns have useful row-group min/max statistics.

In [ ]:
metadata_sql = """
SELECT
    file_name,
    count(DISTINCT row_group_id) AS row_groups,
    max(row_group_num_rows) AS max_row_group_rows,
    sum(row_group_compressed_bytes) AS compressed_bytes
FROM parquet_metadata({parquet_glob})
GROUP BY file_name
ORDER BY file_name
"""

for name, glob in DATASETS.items():
    print('\n##', name)
    try:
        display(con.execute(render(metadata_sql, glob, PARAMS)).df())
    except Exception as error:
        print(error)

In [ ]:
stats_sql = """
SELECT
    path_in_schema,
    count(*) AS row_groups,
    min(stats_min_value) AS global_min,
    max(stats_max_value) AS global_max
FROM parquet_metadata({parquet_glob})
WHERE path_in_schema IN (
    'hash:hash',
    'datetime',
    'collection',
    'id',
    'bbox.xmin',
    'bbox.ymin',
    'bbox.xmax',
    'bbox.ymax'
)
GROUP BY path_in_schema
ORDER BY path_in_schema
"""

for name, glob in DATASETS.items():
    print('\n##', name)
    try:
        display(con.execute(render(stats_sql, glob, PARAMS)).df())
    except Exception as error:
        print(error)

## Query Suite

In [ ]:
QUERIES = {
    'q01_full_dataset_count': """
SELECT count(*)
FROM read_parquet({parquet_glob}, hive_partitioning = false)
""",
    'q02_time_range_count': """
SELECT count(*)
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE datetime >= {start_datetime}
  AND datetime < {end_datetime}
""",
    'q03_bbox_count': """
SELECT count(*)
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE bbox.xmax >= {minx}
  AND bbox.xmin <= {maxx}
  AND bbox.ymax >= {miny}
  AND bbox.ymin <= {maxy}
""",
    'q04_stac_search_count': """
SELECT count(*)
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE collection = {collection}
  AND datetime >= {start_datetime}
  AND datetime < {end_datetime}
  AND bbox.xmax >= {minx}
  AND bbox.xmin <= {maxx}
  AND bbox.ymax >= {miny}
  AND bbox.ymin <= {maxy}
""",
    'q05_hash_range_search': """
SELECT count(*)
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE "hash:hash" BETWEEN {min_hash} AND {max_hash}
  AND datetime >= {start_datetime}
  AND datetime < {end_datetime}
  AND bbox.xmax >= {minx}
  AND bbox.xmin <= {maxx}
  AND bbox.ymax >= {miny}
  AND bbox.ymin <= {maxy}
""",
    'q06_search_page_datetime_order': """
SELECT id, collection, datetime
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE collection = {collection}
  AND datetime >= {start_datetime}
  AND datetime < {end_datetime}
  AND bbox.xmax >= {minx}
  AND bbox.xmin <= {maxx}
  AND bbox.ymax >= {miny}
  AND bbox.ymin <= {maxy}
ORDER BY datetime, id
LIMIT 100
""",
    'q06_search_page_hash_order': """
SELECT id, collection, datetime
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE collection = {collection}
  AND datetime >= {start_datetime}
  AND datetime < {end_datetime}
  AND bbox.xmax >= {minx}
  AND bbox.xmin <= {maxx}
  AND bbox.ymax >= {miny}
  AND bbox.ymin <= {maxy}
ORDER BY "hash:hash", id
LIMIT 100
""",
    'q07_attribute_filter': """
SELECT count(*)
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE collection = {collection}
  AND datetime >= {start_datetime}
  AND datetime < {end_datetime}
  AND bbox.xmax >= {minx}
  AND bbox.xmin <= {maxx}
  AND bbox.ymax >= {miny}
  AND bbox.ymin <= {maxy}
  AND "eo:cloud_cover" <= {max_cloud_cover}
""",
    'q08_exact_geometry_intersects': """
SELECT count(*)
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE datetime >= {start_datetime}
  AND datetime < {end_datetime}
  AND bbox.xmax >= {minx}
  AND bbox.xmin <= {maxx}
  AND bbox.ymax >= {miny}
  AND bbox.ymin <= {maxy}
  AND ST_Intersects(geometry, ST_GeomFromText({aoi_wkt}))
""",
    'q09_grouped_aggregation': """
SELECT
    collection,
    date_trunc('month', datetime) AS month,
    count(*) AS items
FROM read_parquet({parquet_glob}, hive_partitioning = false)
GROUP BY collection, month
ORDER BY collection, month
""",
    'q10_collection_latest_items': """
SELECT id, collection, datetime
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE collection = {collection}
ORDER BY datetime DESC
LIMIT 100
""",
    'q11_specific_id_lookup': """
SELECT id, collection, datetime
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE id = {id}
""",
    'q11_scoped_id_lookup': """
SELECT id, collection, datetime
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE collection = {collection}
  AND id = {id}
""",
}

list(QUERIES)

## Run Benchmarks

This cell runs the full query suite across all dataset variants. Hash-only queries are automatically skipped for the Microsoft dataset because it does not have the `"hash:hash"` column.

Use `median_seconds` for comparisons. `best_seconds` is useful for spotting warm-cache potential, but it can be optimistic.


In [ ]:
HASH_ONLY_QUERIES = {
    'q05_hash_range_search',
    'q06_search_page_hash_order',
}
HASHED_DATASETS = {'hashed_128_files', 'hashed_12_files'}
def run_benchmark_matrix(datasets: dict[str, str], queries: dict[str, str], repeats: int = 5):
    results = []
    for dataset_name, dataset_glob in datasets.items():
        for query_name, template in queries.items():
            if query_name in HASH_ONLY_QUERIES and dataset_name not in HASHED_DATASETS:
                continue
            sql = render(template, dataset_glob, PARAMS)
            try:
                rows, timings = time_query(sql, repeats=repeats)
            except Exception as error:
                print(f'{dataset_name} / {query_name}: {error}')
                continue
            results.append(BenchmarkResult(
                dataset=dataset_name,
                query=query_name,
                rows=rows,
                best_seconds=min(timings),
                median_seconds=statistics.median(timings),
                runs=timings,
            ))
            print(f'{dataset_name} / {query_name}: rows={rows} best={min(timings):0.4f}s median={statistics.median(timings):0.4f}s')
    return results


def results_table(results: list[BenchmarkResult]):
    rows = [
        {
            'query': result.query,
            'dataset': result.dataset,
            'rows': result.rows,
            'best_seconds': result.best_seconds,
            'median_seconds': result.median_seconds,
        }
        for result in results
    ]
    if not rows:
        return pd.DataFrame(columns=['query', 'dataset', 'rows', 'best_seconds', 'median_seconds'])
    return pd.DataFrame(rows).sort_values(['query', 'median_seconds']).reset_index(drop=True)


results = run_benchmark_matrix(DATASETS, QUERIES, repeats=5)
summary = results_table(results)
display(summary)


## Analyze Results

These derived tables make the first-pass interpretation easier. The speedup table compares median runtimes by query. The file summary helps identify whether differences are caused by hash ordering or by lower-level Parquet characteristics such as file size and row-group layout.


In [ ]:
pivot = summary.pivot(index='query', columns='dataset', values='median_seconds')
speedups = pivot.copy()
if {'microsoft', 'hashed_12_files'}.issubset(speedups.columns):
    speedups['microsoft_vs_hashed_12_speedup'] = speedups['microsoft'] / speedups['hashed_12_files']
if {'hashed_128_files', 'hashed_12_files'}.issubset(speedups.columns):
    speedups['hashed_128_vs_hashed_12_speedup'] = speedups['hashed_128_files'] / speedups['hashed_12_files']

display(speedups.reset_index())


In [ ]:
metadata_rows = []
for dataset, glob in DATASETS.items():
    df = con.execute(render(metadata_sql, glob, PARAMS)).df()
    metadata_rows.append({
        'dataset': dataset,
        'files': len(df),
        'row_groups': int(df['row_groups'].sum()),
        'compressed_gb': float(df['compressed_bytes'].sum() / 1_000_000_000),
        'median_row_groups_per_file': float(df['row_groups'].median()),
        'median_max_row_group_rows': float(df['max_row_group_rows'].median()),
    })

file_summary = pd.DataFrame(metadata_rows).sort_values('dataset').reset_index(drop=True)
display(file_summary)


### Initial Reading Checklist

- If `microsoft_vs_hashed_12_speedup` is high for `q04_stac_search_count`, the generated hashed 12-file layout is materially better for the STAC-style query.
- If `hashed_128_vs_hashed_12_speedup` is above 1, the 12-file rewrite is faster than the 128-file layout for that query. That is a file-count or rewrite effect, not a hash effect.
- If `q01_full_dataset_count` is dramatically different, do not attribute all wins to hash sorting. It usually indicates major Parquet-level differences such as compressed size, metadata, row groups, or schema encoding.
- Treat `q05_hash_range_search` as provisional until the hash bounds are computed from the actual query window instead of placeholder min/max values.


## How To Read The Summary

For each query, compare rows with the same `query` value:

- `microsoft` vs `hashed_12_files` isolates sort/layout effects while keeping file count at 12.
- `hashed_128_files` vs `hashed_12_files` shows how much file count changes timing for the same hashed data.
- `q01_full_dataset_count` is mostly a raw scan control. If this differs a lot, file count/compression/schema overhead may be dominating.
- `q04_stac_search_count` is the main STAC-style query to watch. A hashed win here is the strongest signal.
- `q05_hash_range_search` is only meaningful once `PARAMS['min_hash']` and `PARAMS['max_hash']` represent the AOI/time window. Until then, treat it as provisional.
- `q11_specific_id_lookup` is a control. Hash sorting is not expected to help much for a bare id lookup.

Use `median_seconds` for comparison.


## Inspect a Plan

Use this when a timing difference looks interesting. The key line to look for is `Total Files Read`; if that number drops, DuckDB is pruning files. Also inspect filters shown under `TABLE_SCAN`.

Start with `q04_stac_search_count` on `microsoft` and `hashed_12_files`, then compare their plans side by side.


In [ ]:
dataset_name = 'hashed_12_files'
query_name = 'q04_stac_search_count'

sql = render(QUERIES[query_name], DATASETS[dataset_name], PARAMS)
print(sql)
print(explain_analyze(sql))

In [ ]:
# JSON profile for downstream parsing. This is mostly useful once you know which query/dataset pair matters.
# Uncomment when needed.
# profile = explain_analyze_json(sql)
# profile


## Export Results

Run this after `results` exists and you want to save the timings for sharing or comparison.


In [ ]:
def write_results(results: list[BenchmarkResult], path: str | Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = [
        {
            'dataset': result.dataset,
            'query': result.query,
            'rows': result.rows,
            'best_seconds': result.best_seconds,
            'median_seconds': result.median_seconds,
            'runs': list(result.runs),
            'duckdb_version': duckdb.__version__,
        }
        for result in results
    ]
    path.write_text(json.dumps(rows, indent=2), encoding='utf-8')
    return path


write_results(results, '../benchmark-results/duckdb-geoparquet-results.json')
